In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prepare_IEEE import process_all_IEEE_files, load_IEEE_labels
from tsfresh_features_IEEE import extract_tsfresh_features


In [ ]:
WINDOW_SIZE = 1024
OVERLAP = int(WINDOW_SIZE/2)
MAX_WINDOWS = 200  # windows kept per signal per recording (None keeps all ~1950, which is very slow in tsfresh)

all_processed_windows_a, all_processed_windows_b, all_processed_windows_c = process_all_IEEE_files(WINDOW_SIZE, OVERLAP, MAX_WINDOWS)

print(f"Total processed windows for 'isa': {len(all_processed_windows_a)}")
print(f"Total processed windows for 'isb': {len(all_processed_windows_b)}")
print(f"Total processed windows for 'isc': {len(all_processed_windows_c)}")

In [ ]:
all_processed_windows_a[0]

In [ ]:
def windows_to_dataframe(processed_windows):
    rows = []

    for window in processed_windows:
        row = window["metadata"].copy()

        for i, value in enumerate(window["data"]):
            row[f"sample_{i}"] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Step 2: Convert windows to DataFrames

df_a = windows_to_dataframe(all_processed_windows_a)

df_b = windows_to_dataframe(all_processed_windows_b)

df_c = windows_to_dataframe(all_processed_windows_c)

In [ ]:
df_a

In [ ]:
# Step 3: Extract TSFresh features

df_features_a = extract_tsfresh_features(df_a, "ISA")
df_features_b = extract_tsfresh_features(df_b, "ISB")
df_features_c = extract_tsfresh_features(df_c, "ISC")

In [ ]:
df_features_a

In [ ]:
columns_to_drop = ['value__sum_values', 'value__variance', 'value__length']

# Drop columns from each DataFrame
df_features_a = df_features_a.drop(columns=columns_to_drop, errors='ignore')
df_features_b = df_features_b.drop(columns=columns_to_drop, errors='ignore')
df_features_c = df_features_c.drop(columns=columns_to_drop, errors='ignore')

print("Dropped specified columns from df_features_a, df_features_b, and df_features_c.")
print(f"New shape of df_features_a: {df_features_a.shape}")
print(f"New shape of df_features_b: {df_features_b.shape}")
print(f"New shape of df_features_c: {df_features_c.shape}")

In [ ]:
feature_columns = [col for col in df_features_a.columns if not col in ['file_name', 'window_num', 'window_size', 'overlap', 'signal_type']]

feature_rename_map = {
    old_name: f'tsff_{i+1}'
    for i, old_name in enumerate(feature_columns)
}

def rename_df_features(df: pd.DataFrame, phase_suffix: str):
    """Rename feature columns as tsff_<phase>_<number>."""
    df_renamed = df.copy()

    new_columns = {}

    for col in df_renamed.columns:
        if col in feature_rename_map:
            feature_num = feature_rename_map[col].replace('tsff_', '')
            new_columns[col] = f"tsff_{phase_suffix}_{feature_num}"
        else:
            new_columns[col] = col

    return df_renamed.rename(columns=new_columns)

# Apply the renaming to each phase dataframe
df_features_a_renamed = rename_df_features(df_features_a, '_a')
df_features_b_renamed = rename_df_features(df_features_b, '_b')
df_features_c_renamed = rename_df_features(df_features_c, '_c')


In [ ]:
df_features_a_renamed

In [ ]:
# Define common columns for merging (metadata that should be the same across phases for a given window)
common_merge_cols = ['file_name', 'window_num', 'window_size', 'overlap']

# Start with phase A features, selecting only the necessary columns
df_merged = df_features_a_renamed[common_merge_cols + [col for col in df_features_a_renamed.columns if col.startswith('tsff_')]]

# Merge with phase B features
# Select only the features and common merge columns from df_features_b_renamed
df_b_for_merge = df_features_b_renamed[common_merge_cols + [col for col in df_features_b_renamed.columns if col.startswith('tsff_')]]
df_merged = pd.merge(df_merged, df_b_for_merge, on=common_merge_cols, how='inner')

# Merge with phase C features
# Select only the features and common merge columns from df_features_c_renamed
df_c_for_merge = df_features_c_renamed[common_merge_cols + [col for col in df_features_c_renamed.columns if col.startswith('tsff_')]]
df_merged = pd.merge(df_merged, df_c_for_merge, on=common_merge_cols, how='inner')

print(f"Merged DataFrame created with {len(df_merged)} rows and {df_merged.shape[1]} columns.")
display(df_merged.head())

In [ ]:
## ----- MAPPING FILE NAMES TO GROUPS AND LABELS (TARGET) ----##
# class: 1 = healthy, 2 = one broken bar, 3 = two broken bars, ...
# file_group: condition + load level (e.g. 'r1b_torque05'), used for stratified splitting

labels = load_IEEE_labels()

valid_files = labels["file_name"].tolist()
file_to_class_map = dict(zip(labels["file_name"], labels["class"]))
file_to_group_map = dict(zip(labels["file_name"], labels["file_group"]))

display(labels.groupby(["condition", "class"]).size().rename("recordings"))

In [ ]:
# Filtering df_merged for only those embeddings that belong to valid files
df_merged_valid = df_merged[df_merged['file_name'].isin(valid_files)].copy()
print(f"Generated {len(valid_files)} valid files:")


# Attaching target column
df_merged_valid['class'] = df_merged_valid['file_name'].map(file_to_class_map)

# Add a file group column to enable stratified splitting
df_merged_valid['file_group'] = df_merged_valid['file_name'].map(file_to_group_map)
display(df_merged_valid.head())

In [ ]:
# ---- Train-test split at file-level while stratifying on file_group ---- #
file_df = df_merged_valid[['file_name', 'file_group']].drop_duplicates()

from sklearn.model_selection import train_test_split
train_files, test_files = train_test_split(file_df["file_name"],
                                           stratify=file_df["file_group"],
                                           test_size=0.2,
                                           random_state=67)

train_df = df_merged_valid[df_merged_valid["file_name"].isin(train_files)]
test_df = df_merged_valid[df_merged_valid["file_name"].isin(test_files)]

In [ ]:
# Checking this worked.  Should give an empty set
set(train_df["file_name"]) & set(test_df["file_name"])

In [ ]:
print(train_df["file_group"].value_counts(normalize=True))
print(test_df["file_group"].value_counts(normalize=True))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify feature columns (those starting with 'moment_')
feature_cols = [col for col in train_df.columns if col.startswith('tsff_')]

# Prepare training data
X_train = train_df[feature_cols]
y_train = train_df['class']

# Prepare testing data
X_test = test_df[feature_cols]
y_test = test_df['class']

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardized using StandardScaler.")

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np # Import numpy for type hinting np.ndarray
from sklearn.linear_model import LogisticRegression

def train_svc_model(X_train: pd.DataFrame | np.ndarray, y_train: pd.Series, kernel: str = 'rbf', class_weight: str = 'balanced', random_state: int = 42) -> SVC:
  """
  Trains an SVC classifier and returns the trained model.

  Args:
    X_train (pd.DataFrame | np.ndarray): Training features.
    y_train (pd.Series): Training labels.
    kernel (str): Specifies the kernel type to be used in the algorithm.
    class_weight (str): Weights associated with classes.
    random_state (int): The seed of the pseudo random number generator.

  Returns:
    SVC: The trained SVC model.
  """
  model = SVC(kernel=kernel, class_weight=class_weight, random_state=random_state)
  print(f"Training SVM model with {kernel} kernel and balanced class weights...")
  model.fit(X_train, y_train)
  print(f"SVM model training complete with {kernel} kernel.")
  return model

def evaluate_model(y_true: pd.Series, y_pred: np.ndarray, title: str):
  """
  Evaluates a classification model by printing the classification report and confusion matrix.

  Args:
    y_true (pd.Series): True labels.
    y_pred (np.ndarray): Predicted labels.
    title (str): Title for the evaluation output.
  """
  print(f"\nClassification Report ({title}):")
  print(classification_report(y_true, y_pred))
  print(f"\nConfusion Matrix ({title}):")
  print(confusion_matrix(y_true, y_pred))

def train_log_reg_model(X_train: pd.DataFrame | np.ndarray, y_train: pd.Series,
                      solver: str = 'liblinear', penalty: str = 'l2', C: float = 1.0,
                      class_weight: str = 'balanced', random_state: int = 42, max_iter: int = 1000) -> LogisticRegression:
  """
  Trains a Logistic Regression classifier and returns the trained model.

  Args:
    X_train (pd.DataFrame | np.ndarray): Training features.
    y_train (pd.Series): Training labels.
    solver (str): Algorithm to use in the optimization problem.
    penalty (str): Used to specify the norm of the penalty.
    C (float): Inverse of regularization strength.
    class_weight (str): Weights associated with classes.
    random_state (int): The seed of the pseudo random number generator.
    max_iter (int): Maximum number of iterations for the solver to converge.

  Returns:
    LogisticRegression: The trained Logistic Regression model.
  """
  model = LogisticRegression(solver=solver, penalty=penalty, C=C,
                             class_weight=class_weight, random_state=random_state, max_iter=max_iter)
  print(f"Training Logistic Regression model with solver='{solver}', penalty='{penalty}', C={C} and balanced class weights...")
  model.fit(X_train, y_train)
  print("Logistic Regression model training complete.")
  return model

In [ ]:
# Train SVM model with RBF kernel on UNscaled data
svm_model_unscaled_rbf = train_svc_model(X_train, y_train, kernel='rbf')

In [ ]:
# Make predictions on the unscaled test set
y_pred_unscaled_rbf = svm_model_unscaled_rbf.predict(X_test)

# Evaluate the model
evaluate_model(y_test, y_pred_unscaled_rbf, "RBF Kernel SVM on Unscaled Data")

In [ ]:
# Train SVM model with RBF kernel on scaled data
svm_model_scaled_rbf = train_svc_model(X_train_scaled, y_train, kernel='rbf')

In [ ]:
# Make predictions on the scaled test set
y_pred_scaled_rbf = svm_model_scaled_rbf.predict(X_test_scaled)

# Evaluate the model
evaluate_model(y_test, y_pred_scaled_rbf, "RBF Kernel SVM on Scaled Data")